# Export YOLOv8n to TensorRT

This notebook exports YOLOv8n to ONNX and builds `yolov8_trt/1/model.plan` with `trtexec`.

Build this TensorRT plan in the same Triton image and on the same GPU class that will serve it.

In [ ]:
%pip install --no-cache-dir "ultralytics==8.4.0" "onnx==1.18.0"
%pip uninstall -y opencv-python opencv-contrib-python
%pip install --no-cache-dir "opencv-python-headless==4.12.0.88"

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("USER", "workspace")
os.environ.setdefault("LOGNAME", "workspace")

from ultralytics import YOLO

repo = Path.cwd()
engine_path = repo / "yolov8_trt" / "1" / "model.plan"
engine_path.parent.mkdir(parents=True, exist_ok=True)

model = YOLO("yolov8n.pt")
onnx_path = Path(model.export(format="onnx", imgsz=640, dynamic=False, simplify=True, opset=17))
target_onnx = repo / "yolov8n.onnx"
if onnx_path.resolve() != target_onnx.resolve():
    target_onnx.write_bytes(onnx_path.read_bytes())

print(f"ONNX: {target_onnx}")
print(f"TensorRT plan target: {engine_path}")

In [ ]:
import os
import subprocess
from pathlib import Path

ENABLE_FP16 = False

trtexec_env = os.environ.copy()
cuda_visible_devices = trtexec_env.get("CUDA_VISIBLE_DEVICES")
if cuda_visible_devices is not None and cuda_visible_devices.strip() in {"", "-1", "none", "None", "void"}:
    trtexec_env.pop("CUDA_VISIBLE_DEVICES", None)
    print(f"Removed CUDA_VISIBLE_DEVICES={cuda_visible_devices!r} for trtexec.")
else:
    print(f"CUDA_VISIBLE_DEVICES={cuda_visible_devices!r}")
print(f"NVIDIA_VISIBLE_DEVICES={trtexec_env.get('NVIDIA_VISIBLE_DEVICES')!r}")

help_result = subprocess.run(
    ["trtexec", "--help"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=trtexec_env,
)
trtexec_help = help_result.stdout

gpu_check = subprocess.run(
    ["nvidia-smi", "-L"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=trtexec_env,
)
if gpu_check.returncode != 0:
    raise RuntimeError(
        "No CUDA-capable GPU is visible in this workspace. "
        "Create or restart the Development workspace with GPU count 1, then rerun this cell.\n\n"
        f"nvidia-smi output:\n{gpu_check.stdout}"
    )
print(gpu_check.stdout.strip())

cmd = [
    "trtexec",
    "--onnx=yolov8n.onnx",
    "--saveEngine=yolov8_trt/1/model.plan",
]
if "--buildOnly" in trtexec_help:
    cmd.append("--buildOnly")
elif "--skipInference" in trtexec_help:
    cmd.append("--skipInference")

if ENABLE_FP16:
    if "--fp16" in trtexec_help:
        cmd.append("--fp16")
    else:
        print("ENABLE_FP16 requested, but this trtexec does not support --fp16; building FP32.")

print("Running:", " ".join(cmd))
log_path = Path("trtexec.log")
result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=trtexec_env)
log_path.write_text(result.stdout)
print(f"trtexec log written to {log_path.resolve()}")
interesting = [
    line for line in result.stdout.splitlines()
    if "[E]" in line or "[W]" in line or "ERROR" in line or "failed" in line.lower()
]
print("\n".join(interesting[-40:]) or result.stdout[-4000:])
result.check_returncode()

In [ ]:
from pathlib import Path

plan = Path("yolov8_trt/1/model.plan")
assert plan.exists(), "TensorRT plan was not created"
print(f"Created {plan} ({plan.stat().st_size / (1024 * 1024):.1f} MiB)")